<div style="border-left:6px solid #2f9e41; padding:8px 14px; background:rgba(47,158,65,0.08)">
<b>IFRJ — Campus Paracambi</b> · Engenharia Mecânica<br>
Oficina de Simulação Computacional em Ciências Térmicas · <b>Prof. Luiz Ferreira</b>
</div>

# Dia 1 — Condução de calor e métodos clássicos de solução

Neste primeiro dia você vai **simular a condução de calor** numa barra, numa parede e numa placa, e conhecer os **métodos clássicos** que estão por trás de qualquer programa de simulação: diferenças finitas, métodos explícito e implícito, algoritmo de Thomas, Jacobi, Gauss–Seidel e SOR.

Você **não precisa programar nada do zero**. O código já está pronto e comentado. Seu trabalho é **ler, rodar, prever, mudar e comparar** e, nas tarefas ✍️, **escrever** a linha principal de cada método.

| Parte | Assunto | Tempo |
|---|---|---|
| 0 | O mínimo de Python para esta oficina (leitura prévia) | — |
| A | Do contínuo ao discreto: diferenças finitas | 15 min |
| B | Condução 1D transiente: método explícito | 35 min |
| C | Método implícito e algoritmo de Thomas | 25 min |
| D | Condução 2D: transiente e regime permanente | 40 min |

**Novidade: tarefas ✍️.** Em cada parte há uma tarefa em que **você escreve** a linha principal do método (de 1 a 3 linhas). A função `confere_...()` diz na hora se está certo e, se não estiver, qual é o erro mais provável. Logo abaixo de cada tarefa há uma célula **👀 Resposta**, recolhida: abra se travar. Ninguém fica para trás: enquanto a sua linha não passar, os experimentos usam a resposta; quando passar, usam o **seu** código.

**Extra de bioengenharia:** o Experimento B5 (fim da Parte B) usa o mesmo método para simular uma bolsa de gelo num músculo, com a equação do biocalor.

## Como usar este notebook

- Cada bloco cinza é uma **célula de código**. Clique nela e aperte **Shift + Enter** para executar.
- Execute as células **de cima para baixo**: as de baixo usam funções criadas nas de cima. Se aparecer `NameError`, provavelmente uma célula anterior não foi executada. Use *Ambiente de execução → Executar tudo* para recomeçar.
- **Se a sua linha numa tarefa der erro de Python** (um parêntese faltando, por exemplo), o *Executar tudo* para nela. Corrija ou apague a linha, rode a célula 👀 Resposta logo abaixo e continue executando as células seguintes (no Colab, menu *Ambiente de execução*, opção de executar as células depois da atual; no JupyterLite, *Run → Run Selected Cell and All Below*).
- O índice (ícone ☰ à esquerda, no Colab) mostra os títulos deste notebook: use-o para navegar.
- Para guardar suas mudanças: *Arquivo → Salvar uma cópia no Drive*.
- **Sem conta Google (JupyterLite, no navegador):** o notebook abre direto pelo endereço passado pelo professor. Para executar tudo, use *Run → Run All Cells*; o índice fica no painel da esquerda. Suas mudanças ficam guardadas no navegador daquele computador; para levar o arquivo, use *File → Download*.

**Como funcionam os experimentos.** Cada experimento tem uma lista chamada `casos`. Cada caso é uma linha com os parâmetros de uma simulação. O código roda todos os casos e desenha os resultados juntos, para você comparar. Para testar outra situação, **mude um número da lista** (ou copie uma linha e altere) e rode a célula de novo. Antes de rodar, tente **prever** o que vai acontecer: é aí que se aprende.

## Parte 0 — O mínimo de Python para esta oficina

Leia esta parte antes do primeiro encontro (ou consulte quando tiver dúvida). Ela mostra só o que aparece nos códigos da oficina.

### 0.1 Variáveis, contas e `print`

In [ ]:
# O símbolo # inicia um COMENTÁRIO: o Python ignora tudo o que vem depois dele na linha.
# Comentários servem para explicar o código a quem lê.

k = 237.0        # condutividade térmica do alumínio [W/(m·K)]
rho = 2702.0     # massa específica [kg/m³]
cp = 903.0       # calor específico [J/(kg·K)]

alpha = k / (rho * cp)      # difusividade térmica [m²/s]

# print mostra valores na tela. O f antes das aspas ("f-string") permite colocar
# variáveis entre chaves {}; o :.3e formata em notação científica com 3 casas.
print(f"Difusividade do alumínio: alpha = {alpha:.3e} m²/s")

### 0.2 Listas, dicionários e o laço `for`

Os experimentos da oficina usam exatamente esta estrutura: uma **lista** de **dicionários**, percorrida por um laço `for`.

In [ ]:
# LISTA: vários valores entre colchetes, separados por vírgula.
espessuras = [0.01, 0.02, 0.05]

# DICIONÁRIO: pares "chave": valor entre chaves. Acessamos um valor pela chave: material["k"].
material = {"nome": "alumínio", "k": 237.0}
print(material["nome"], "tem k =", material["k"])

# LISTA DE DICIONÁRIOS: cada item é um "caso" com seus parâmetros.
casos = [
    {"nome": "alumínio", "k": 237.0},
    {"nome": "aço inox", "k": 14.9},
]

# O laço for repete o bloco recuado (indentado) uma vez para cada item da lista.
# Em Python, o RECUO (4 espaços) é o que define o que está dentro do laço.
for caso in casos:
    resistencia = 0.02 / caso["k"]     # resistência térmica de uma parede de 2 cm [m²K/W]
    print(f"{caso['nome']:>10}: R = {resistencia:.2e} m²K/W")

### 0.3 Funções

Uma função agrupa um cálculo que vamos repetir. Ela recebe **argumentos** e devolve um resultado com `return`. Os "motores" das simulações desta oficina são funções.

In [ ]:
def numero_de_biot(h, L, k):
    """Texto entre três aspas logo abaixo do def é a 'docstring': explica o que a função faz.
    Biot = resistência de condução interna / resistência de convecção externa."""
    return h * L / k

# Chamando a função com argumentos nomeados (fica mais legível):
Bi = numero_de_biot(h=50.0, L=0.01, k=237.0)
print(f"Bi = {Bi:.4f}")

### 0.4 NumPy: vetores e fatias

O **NumPy** guarda números em vetores (*arrays*) e faz contas em todos os elementos de uma vez. As fatias (*slices*) são a peça mais importante da oficina: com elas escrevemos as fórmulas de diferenças finitas para todos os nós de uma vez.

| Fatia | Elementos | Em diferenças finitas |
|---|---|---|
| `T[1:-1]` | do 2º ao penúltimo | os nós internos $T_i$ |
| `T[2:]` | do 3º ao último | os vizinhos da direita $T_{i+1}$ |
| `T[:-2]` | do 1º ao antepenúltimo | os vizinhos da esquerda $T_{i-1}$ |
| `T[0]`, `T[-1]` | primeiro e último | as duas bordas (em Python, índice `-1` = último elemento) |

In [ ]:
import numpy as np                 # "np" é o apelido padrão do NumPy

T = np.array([10.0, 12.0, 15.0, 19.0, 24.0])   # um vetor com 5 temperaturas
print("T[1:-1] =", T[1:-1])     # nós internos
print("T[2:]   =", T[2:])       # vizinhos da direita, alinhados com T[1:-1]
print("T[:-2]  =", T[:-2])      # vizinhos da esquerda, alinhados com T[1:-1]

# A segunda diferença em TODOS os nós internos, numa linha só e sem laço:
print("T[2:] - 2*T[1:-1] + T[:-2] =", T[2:] - 2 * T[1:-1] + T[:-2])

x = np.linspace(0.0, 1.0, 5)    # 5 pontos igualmente espaçados de 0 a 1
print("x =", x)

In [ ]:
# CUIDADO: "B = A" NÃO copia um vetor. A e B passam a ser o mesmo objeto.
A = np.array([1.0, 2.0, 3.0])
B = A            # mesmo vetor, com outro nome
C = A.copy()     # cópia de verdade
A[0] = 99.0
print("A =", A, "| B =", B, "| C =", C)   # B mudou junto com A; C não

### 0.5 Gráficos com Matplotlib

In [ ]:
import matplotlib.pyplot as plt    # "plt" é o apelido padrão
import time                         # para medir tempo de execução

# Ajustes de aparência usados em toda a oficina
plt.rcParams.update({"figure.figsize": (8, 3.6), "axes.grid": True, "grid.alpha": 0.3})

x = np.linspace(0, 2 * np.pi, 100)
fig, ax = plt.subplots()               # fig = a figura; ax = os eixos onde desenhamos
ax.plot(x, np.sin(x), label="sen(x)")  # label = texto que aparece na legenda
ax.plot(x, np.cos(x), "--", label="cos(x)")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend()
plt.show()

### ⚙️ O verificador das tarefas

Rode a célula abaixo (ela fica recolhida; não precisa abrir). Ela cria as funções `confere_...()`, que conferem o código que você escrever nas **tarefas ✍️** e explicam, em português, o erro mais provável.

In [ ]:
#@title ⚙️ Verificador das tarefas do Dia 1 (rode esta célula; não precisa abrir)
# Cria as funções confere_...(), que testam o código das Tarefas ✍️ e explicam o erro em português.
# Se a sua função passa no teste, os experimentos seguintes usam o SEU código; se não, usam a resposta.
import numpy as np
import matplotlib.pyplot as plt

_SEU, _REF = {}, {}
_FALTA = "⏳ Ainda falta completar: troque o None pela fórmula (mantendo o return) e rode a célula de novo."

def _usar(nome):
    """Função do aluno, se passou no verificador; senão, a resposta."""
    return _SEU.get(nome, _REF[nome])

def _br(valor, formato):
    return format(valor, formato).replace(".", ",")

def _roda(func, *args):
    """Chama a função do aluno e devolve (resultado, problema). problema = None se rodou bem."""
    try:
        res = func(*args)
    except Exception as e:
        m = str(e)
        if "NoneType" in m or "dtype('O')" in m:
            return None, _FALTA
        if "broadcast" in m:
            return None, ("✘ As fatias não estão alinhadas (têm tamanhos diferentes). Os vizinhos dos nós internos "
                          "T[1:-1] são T[2:] e T[:-2]; em 2D, T[1:-1, 2:], T[1:-1, :-2], T[2:, 1:-1] e T[:-2, 1:-1].")
        return None, f"✘ O Python deu um erro ao rodar a sua função: {type(e).__name__}: {e}"
    if res is None:
        return None, _FALTA
    try:
        if np.any(np.isnan(np.asarray(res, dtype=float))):
            return None, _FALTA
    except (TypeError, ValueError):
        return None, _FALTA
    return np.asarray(res, dtype=float), None

def _fim(nome, func, ok, msg, resposta):
    if resposta:
        print(("✔ Esta é a resposta. " if ok else "⚠ ") + msg)
    elif ok:
        _SEU[nome] = func
        print("✔ Certo! " + msg + "\n   Os experimentos abaixo vão usar o SEU código.")
    else:
        _SEU.pop(nome, None)
        print(msg + "\n   Enquanto isso, os experimentos usam a resposta (célula recolhida logo abaixo).")

def _igual(a, b, tol=1e-9):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return a.shape == b.shape and np.allclose(a, b, atol=tol, rtol=1e-9)

# ------------------------------------------------------------------ respostas de referência
def _ref_para_tras(f, x0, dx): return (f(x0) - f(x0 - dx)) / dx
def _ref_ftcs(T, r):
    T[1:-1] = T[1:-1] + r * (T[2:] - 2 * T[1:-1] + T[:-2]); return T
def _ref_sangue(T, omega, rho_c, rho_c_sangue, T_arterial, q_met):
    return (omega * rho_c_sangue * (T_arterial - T) + q_met) / rho_c
def _ref_lado_direito(T, r):
    d = T[1:-1].copy(); d[0] += r * T[0]; d[-1] += r * T[-1]; return d
def _ref_media_jacobi(Ta): return 0.25 * (Ta[1:-1, 2:] + Ta[1:-1, :-2] + Ta[2:, 1:-1] + Ta[:-2, 1:-1])
def _ref_sor(T_no, media, omega): return T_no + omega * (media - T_no)
_REF.update({"para_tras": _ref_para_tras, "ftcs": _ref_ftcs, "sangue": _ref_sangue,
             "lado_direito": _ref_lado_direito, "media_jacobi": _ref_media_jacobi, "sor": _ref_sor})

# ------------------------------------------------------------------ Tarefa A
def confere_para_tras(func, resposta=False):
    passos = np.array([0.2, 0.1, 0.05, 0.025, 0.0125])
    valores = []
    for h in passos:
        v, problema = _roda(func, np.sin, 1.0, h)
        if problema: return _fim("para_tras", func, False, problema, resposta)
        valores.append(float(v))
    valores = np.array(valores); exato = np.cos(1.0); erros = np.abs(valores - exato)
    certo = np.array([_ref_para_tras(np.sin, 1.0, h) for h in passos])
    frente = (np.sin(1.0 + passos) - np.sin(1.0)) / passos
    central = (np.sin(1.0 + passos) - np.sin(1.0 - passos)) / (2 * passos)
    if _igual(valores, certo, 1e-12):
        ordem = np.polyfit(np.log(passos), np.log(erros), 1)[0]
        fig, ax = plt.subplots(figsize=(5, 3.2))
        ax.loglog(passos, erros, "o-", label="sua diferença para trás")
        ax.loglog(passos, 0.5 * passos, "k:", label="inclinação 1")
        ax.set_xlabel("Δx"); ax.set_ylabel("erro"); ax.legend(fontsize=8); plt.show()
        return _fim("para_tras", func, True, f"Ordem medida: {_br(ordem, '.2f')} (1ª ordem, como a diferença para frente).", resposta)
    if np.all(erros < 1e-12):
        msg = "✘ Você usou a derivada exata (cos). A ideia é ESTIMAR a derivada só com f(x0) e f(x0 − dx)."
    elif _igual(valores, frente, 1e-12):
        msg = "✘ Essa é a diferença para FRENTE (usa f(x0 + dx)). A tarefa pede a para trás: f(x0) e f(x0 − dx)."
    elif _igual(valores, central, 1e-12):
        msg = "✘ Essa é a diferença CENTRAL (2ª ordem). A tarefa pede a para trás: f(x0) e f(x0 − dx), dividido por dx."
    elif _igual(valores, -certo, 1e-12):
        msg = "✘ O sinal saiu trocado: é f(x0) − f(x0 − dx), nessa ordem."
    elif _igual(valores, certo * passos, 1e-12):
        msg = "✘ Faltou dividir por dx."
    else:
        msg = (f"✘ Com Δx = 0,0125 o resultado deu {_br(valores[-1], '.4f')}, mas a derivada exata é "
               f"cos(1) ≈ 0,5403. Confira a fórmula: (f(x0) − f(x0 − dx)) / dx.")
    _fim("para_tras", func, False, msg, resposta)

# ------------------------------------------------------------------ Tarefa B
def confere_ftcs(func, resposta=False):
    T0 = np.array([0.0, 100.0, 100.0, 100.0, 0.0]); r = 0.4
    v, problema = _roda(func, T0.copy(), r)
    if problema: return _fim("ftcs", func, False, problema, resposta)
    certo = np.array([0, 60, 100, 60, 0.0])
    if _igual(v, certo):
        L, a, nx = 0.10, 97.1e-6, 21
        dx = L / (nx - 1); dt = r * dx**2 / a; n = int(round(5.0 / dt)); x = np.linspace(0, L, nx)
        T = np.full(nx, 100.0); T[0] = T[-1] = 0.0
        for _ in range(n):
            T = np.asarray(func(T, r), dtype=float)
        try:
            erro = np.max(np.abs(T - solucao_analitica_barra(x, n * dt, L, a, 100.0, 0.0)))
            extra = f" A barra de alumínio, simulada por 5 s, ficou a {_br(erro, '.3f')} °C da série de Fourier."
        except NameError:
            erro, extra = 0.0, ""
        return _fim("ftcs", func, erro < 0.5,
                    "O primeiro passo do Exemplo B.1 da apostila deu [0; 60; 100; 60; 0]." + extra, resposta)
    s = "; ".join(_br(t, "g") for t in v)
    if v.shape != certo.shape:
        msg = "✘ A função mudou o tamanho do vetor T. Altere só os nós internos, T[1:-1]."
    elif v[0] != 0 or v[-1] != 0:
        msg = f"✘ As bordas mudaram (deu [{s}]). Elas são fixas: altere só T[1:-1]."
    elif _igual(v, [0, 140, 100, 140, 0]):
        msg = "✘ A temperatura subiu perto das bordas frias, onde deveria cair: confira o sinal (T + r·(…))."
    elif _igual(v, [0, 100, 140, 100, 0]):
        msg = "✘ Faltou o 2 no termo do meio: T[2:] − 2·T[1:-1] + T[:-2]."
    elif _igual(v, [0, -40, 0, -40, 0]):
        msg = "✘ Faltou somar o valor atual: o novo valor é T[1:-1] + r·(…), não só r·(…)."
    elif _igual(v, [0, 0, 100, 0, 0]):
        msg = "✘ Faltou multiplicar a segunda diferença por r."
    else:
        msg = (f"✘ No Exemplo B.1 da apostila (T = [0; 100; 100; 100; 0], r = 0,4), o primeiro passo dá "
               f"[0; 60; 100; 60; 0]; a sua função deu [{s}].")
    _fim("ftcs", func, False, msg, resposta)

# ------------------------------------------------------------------ Tarefa B5 (bioengenharia)
def confere_sangue(func, resposta=False):
    T = np.array([10.0, 25.0, 37.0]); p = (5e-4, 3.6e6, 3.6e6, 37.0, 700.0)
    v, problema = _roda(func, T.copy(), *p)
    if problema: return _fim("sangue", func, False, problema, resposta)
    certo = _ref_sangue(T, *p)
    if _igual(v, certo, 1e-12):
        nx, L, k, rho_c = 31, 0.03, 0.5, 3.6e6
        dx = L / (nx - 1); alpha = k / rho_c; n = int(np.ceil(3 * 3600 / (0.4 * dx**2 / alpha)))
        dt = 3 * 3600 / n; r = alpha * dt / dx**2; x = np.linspace(0, L, nx)
        Tm = np.full(nx, 37.0); Tm[0] = 5.0
        for _ in range(n):
            Tm[1:-1] = Tm[1:-1] + r * (Tm[2:] - 2 * Tm[1:-1] + Tm[:-2]) + dt * np.asarray(func(Tm[1:-1], *p), float)
        try:
            dif = np.max(np.abs(Tm - biocalor_permanente(x, 5e-4)))
            extra = f" Depois de 3 h simuladas, o músculo ficou a {_br(dif, '.3f')} °C da solução exata do regime permanente."
        except NameError:
            dif, extra = 0.0, ""
        return _fim("sangue", func, dif < 0.05, "O termo do sangue está correto." + extra, resposta)
    if _igual(v, certo * 3.6e6, 1e-6):
        msg = "✘ Faltou dividir por ρc (rho_c): o termo precisa estar em K/s."
    elif _igual(v, (p[0] * p[2] * (T - p[3]) + p[4]) / p[1], 1e-12):
        msg = "✘ O sangue chega a 37 °C e deve AQUECER o tecido frio: use (T_arterial − T), nessa ordem."
    elif _igual(v, p[0] * p[2] * (p[3] - T) / p[1], 1e-12):
        msg = "✘ Faltou o calor do metabolismo, q_met."
    elif _igual(v, (p[2] * (p[3] - T) + p[4]) / p[1], 1e-12):
        msg = "✘ Faltou multiplicar pela perfusão, omega."
    else:
        msg = (f"✘ Para T = 10 °C, o termo deveria dar {_br(certo[0], '.5f')} K/s, e a sua função deu "
               f"{_br(v.ravel()[0], '.5g')}. Confira: (omega · rho_c_sangue · (T_arterial − T) + q_met) / rho_c.")
    _fim("sangue", func, False, msg, resposta)

# ------------------------------------------------------------------ Tarefa C
def confere_lado_direito(func, resposta=False):
    T = np.array([10.0, 50.0, 60.0, 70.0, 30.0]); r = 0.5; Tc = T.copy()
    v, problema = _roda(func, Tc, r)
    if problema: return _fim("lado_direito", func, False, problema, resposta)
    if not np.array_equal(Tc, T):
        return _fim("lado_direito", func, False, "✘ A sua função alterou T! Sem o .copy(), d e T viram o mesmo vetor (Parte 0.4).", resposta)
    certo = [55, 60, 85]
    if _igual(v, certo):
        L, a, nx = 0.10, 97.1e-6, 51
        dx = L / (nx - 1); dt = dx**2 / (2 * a); r2 = a * dt / dx**2; n = int(round(10.0 / dt)); x = np.linspace(0, L, nx)
        Tb = np.full(nx, 100.0); Tb[0] = Tb[-1] = 0.0; m = nx - 2
        for _ in range(n):
            Tb[1:-1] = thomas(np.full(m, -r2), np.full(m, 1 + 2 * r2), np.full(m, -r2), np.asarray(func(Tb, r2), float))
        erro = np.max(np.abs(Tb - solucao_analitica_barra(x, n * dt, L, a, 100.0, 0.0)))
        return _fim("lado_direito", func, erro < 0.2,
                    f"Com o seu lado direito, o implícito ficou a {_br(erro, '.3f')} °C da série de Fourier em t = 10 s.", resposta)
    s = "; ".join(_br(t, "g") for t in v)
    if _igual(v, [45, 60, 55]):
        msg = "✘ Sinal trocado: a borda conhecida passa para o lado direito SOMANDO r·T_borda."
    elif _igual(v, [60, 60, 100]):
        msg = "✘ Faltou multiplicar a temperatura da borda por r."
    elif _igual(v, [65, 60, 75]):
        msg = "✘ As bordas estão trocadas: d[0] recebe a borda esquerda, T[0]; d[-1] recebe a direita, T[-1]."
    elif _igual(v, [50, 60, 70]):
        msg = "✘ As bordas não entraram: some r·T[0] em d[0] e r·T[-1] em d[-1]."
    else:
        msg = f"✘ Para T = [10; 50; 60; 70; 30] e r = 0,5, o lado direito deveria ser [55; 60; 85]; deu [{s}]."
    _fim("lado_direito", func, False, msg, resposta)

# ------------------------------------------------------------------ Tarefa D
def _placa_teste(n=21):
    T = np.full((n, n), 20.0); T[-1, 1:-1] = 100.0; return T

def confere_jacobi(func, resposta=False):
    A = np.random.default_rng(1).uniform(0, 100, (5, 6))
    v, problema = _roda(func, A.copy())
    if problema: return _fim("media_jacobi", func, False, problema, resposta)
    certo = _ref_media_jacobi(A)
    if v.shape != certo.shape:
        return _fim("media_jacobi", func, False, f"✘ A matriz devolvida tem tamanho {v.shape}, mas deveria ter o tamanho "
                    f"dos nós internos, {certo.shape}. Use as fatias T_antigo[1:-1, 2:] etc.", resposta)
    if _igual(v, certo):
        T = _placa_teste()
        for it in range(1, 50001):
            Ta = T.copy(); T[1:-1, 1:-1] = np.asarray(func(Ta), float)
            if np.max(np.abs(T - Ta)) < 1e-4: break
        return _fim("media_jacobi", func, True, f"Na placa 21 × 21 do Experimento D2: {it} iterações e centro a "
                    f"{_br(T[10, 10], '.2f')} °C (o valor exato é 40 °C).", resposta)
    if _igual(v, 4 * certo):
        msg = "✘ Faltou dividir por 4 (ou multiplicar por 0,25): é a MÉDIA dos quatro vizinhos."
    else:
        msg = ("✘ O resultado não é a média dos quatro vizinhos. Confira as quatro fatias: T_antigo[1:-1, 2:] (direita), "
               "T_antigo[1:-1, :-2] (esquerda), T_antigo[2:, 1:-1] (cima) e T_antigo[:-2, 1:-1] (baixo).")
    _fim("media_jacobi", func, False, msg, resposta)

def confere_sor(func, resposta=False):
    testes = [(20.0, 40.0, 1.0), (20.0, 40.0, 1.5), (50.0, 30.0, 1.8)]
    vals = []
    for t in testes:
        v, problema = _roda(func, *t)
        if problema: return _fim("sor", func, False, problema, resposta)
        vals.append(float(v))
    certo = [40.0, 50.0, 14.0]
    if _igual(vals, certo):
        n = 21; T = _placa_teste(n); w = 2 / (1 + np.sin(np.pi / (n - 1)))
        for it in range(1, 5001):
            maior = 0.0
            for j in range(1, n - 1):
                for i in range(1, n - 1):
                    media = 0.25 * (T[j, i + 1] + T[j, i - 1] + T[j + 1, i] + T[j - 1, i])
                    novo = float(func(T[j, i], media, w)); maior = max(maior, abs(novo - T[j, i])); T[j, i] = novo
            if maior < 1e-4: break
        return _fim("sor", func, True, f"Com o ω ótimo ({_br(w, '.2f')}), o SOR convergiu em {it} iterações na placa 21 × 21.", resposta)
    if _igual(vals, [40.0, 40.0, 30.0]):
        msg = "✘ Com ω = 1,5 o valor novo deveria passar da média (dar 50), mas deu 40: faltou o ω."
    elif _igual(vals, [0.0, -10.0, 86.0]):
        msg = "✘ Sinal trocado: o valor novo é T_no + ω·(média − T_no)."
    elif _igual(vals, [40.0, 60.0, 54.0]):
        msg = "✘ O ω multiplica a CORREÇÃO (média − T_no), não a média: T_no + ω·(média − T_no)."
    else:
        msg = (f"✘ Para T_no = 20, média = 40 e ω = 1,5, o valor novo deveria ser 50; a sua função deu "
               f"{_br(vals[1], 'g')}. Confira: T_no + omega·(media − T_no).")
    _fim("sor", func, False, msg, resposta)

print("Verificador pronto: as funções confere_...() estão carregadas.")

## Parte A — Do contínuo ao discreto: diferenças finitas

O computador não guarda uma função contínua $T(x)$: guarda os valores dela em pontos separados por $\Delta x$. Para resolver uma equação diferencial, trocamos cada derivada por uma conta com esses valores. A série de Taylor diz quanto erramos:

$$\text{para frente:}\quad T'(x)\approx\frac{T(x+\Delta x)-T(x)}{\Delta x}\quad(\text{erro}\propto\Delta x)$$

$$\text{central:}\quad T'(x)\approx\frac{T(x+\Delta x)-T(x-\Delta x)}{2\,\Delta x}\quad(\text{erro}\propto\Delta x^{2})$$

$$\text{segunda derivada:}\quad T''(x)\approx\frac{T(x+\Delta x)-2\,T(x)+T(x-\Delta x)}{\Delta x^{2}}\quad(\text{erro}\propto\Delta x^{2})$$

A potência de $\Delta x$ no erro é a **ordem** do método. Vamos medi-la.

### Experimento A1 — o erro diminui como a teoria prevê?

Estimamos a derivada de $\text{sen}(x)$ em $x=1$, cuja resposta exata é $\cos(1)$, com vários $\Delta x$.

**Antes de rodar:** se $\Delta x$ cair pela metade, o erro da diferença para frente cai quantas vezes? E o da central?

In [ ]:
def f(x):
    """Função de teste: conhecemos a derivada exata, então podemos medir o erro."""
    return np.sin(x)

def derivada_exata(x):
    return np.cos(x)

x0 = 1.0    # ponto onde estimamos a derivada

# ===== EXPERIMENTO A1: mude ou acrescente valores de Δx =====
passos_dx = [0.2, 0.1, 0.05, 0.025, 0.0125]

erros_frente = []     # listas vazias; .append() acrescenta um valor no fim
erros_central = []
for dx in passos_dx:
    para_frente = (f(x0 + dx) - f(x0)) / dx
    central = (f(x0 + dx) - f(x0 - dx)) / (2 * dx)
    erros_frente.append(abs(para_frente - derivada_exata(x0)))    # abs = valor absoluto
    erros_central.append(abs(central - derivada_exata(x0)))

print("   Δx      erro (frente)   erro (central)")
for dx, e1, e2 in zip(passos_dx, erros_frente, erros_central):   # zip percorre as listas juntas
    print(f"{dx:7.4f}   {e1:12.2e}   {e2:12.2e}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(passos_dx, erros_frente, "o-", label="para frente")
ax.loglog(passos_dx, erros_central, "s-", label="central")
dx_ref = np.array(passos_dx)
ax.loglog(dx_ref, 0.5 * dx_ref, "k:", label="inclinação 1 (1ª ordem)")
ax.loglog(dx_ref, 0.2 * dx_ref**2, "k--", label="inclinação 2 (2ª ordem)")
ax.set_xlabel("Δx"); ax.set_ylabel("erro"); ax.legend()
plt.show()

**O que observar.** Num gráfico log-log, a ordem é a **inclinação** da reta. A diferença para frente segue a reta de inclinação 1 e a central, a de inclinação 2. Medir essa inclinação ao refinar a malha é a forma profissional de **verificar** um código.

**Sua vez:** troque `x0` por 2,0 ou a função por `np.exp(x)` (e a derivada exata também). As inclinações mudam?

*Para saber mais:* Chapra e Canale, *Métodos Numéricos para Engenharia*, cap. 4 (série de Taylor e erros); livro colaborativo de Cálculo Numérico da UFRGS (gratuito).

### ✍️ Tarefa A — a diferença para trás

A tabela da apostila (seção A.1) traz uma terceira fórmula, a diferença **para trás**: $T'_i\approx(T_i-T_{i-1})/\Delta x$. Complete a linha abaixo e rode. O verificador mede a ordem do erro com os mesmos $\Delta x$ do Experimento A1.

**Antes de rodar:** a ordem vai ser 1 ou 2? Se travar, abra a resposta logo abaixo.

In [ ]:
def minha_para_tras(f, x0, dx):
    return None   # ← troque None pela diferença para trás, usando f(x0) e f(x0 - dx)

confere_para_tras(minha_para_tras)

👀 **Resposta da Tarefa A:** a célula abaixo está recolhida. Abra (clique nela ou em *Mostrar código*) só se travar, e rode.

In [ ]:
#@title 👀 Resposta da Tarefa A (abra se travar)
def para_tras_resposta(f, x0, dx):
    return (f(x0) - f(x0 - dx)) / dx

confere_para_tras(para_tras_resposta, resposta=True)

## Parte B — Condução 1D transiente: método explícito

### B.1 O modelo e o esquema

Para uma barra com propriedades constantes e sem geração, o balanço de energia com a lei de Fourier dá a **equação do calor**:

$$\frac{\partial T}{\partial t}=\alpha\,\frac{\partial^{2}T}{\partial x^{2}},\qquad \alpha=\frac{k}{\rho\,c_p}$$

Com a derivada no tempo para frente e a segunda derivada central, obtemos o **método explícito** (FTCS):

$$T_i^{n+1}=T_i^{n}+r\left(T_{i+1}^{n}-2T_i^{n}+T_{i-1}^{n}\right),\qquad r=\frac{\alpha\,\Delta t}{\Delta x^{2}}$$

Tudo do lado direito é conhecido no instante $n$: basta "marchar" no tempo. O número $r$ (número de Fourier da malha) decide a estabilidade: o método só funciona se $r\le 1/2$.

### B.2 O motor da simulação

As duas funções abaixo são o "motor": a solução numérica e a solução analítica (série de Fourier), que usamos para conferir. **Não é preciso alterá-las**, só executar. A linha do FTCS ficou de fora de propósito: você vai escrevê-la na **Tarefa B**, logo depois.

In [ ]:
def solucao_analitica_barra(x, t, L, alpha, T_inicial, T_borda, n_termos=200):
    """Solução exata (série de Fourier) da barra que começa toda a T_inicial
    e tem as duas pontas mantidas a T_borda a partir de t = 0."""
    soma = np.zeros_like(x)                  # vetor de zeros do mesmo tamanho de x
    for n in range(1, 2 * n_termos, 2):      # n = 1, 3, 5, ... (os termos pares são nulos)
        soma += (4 / (n * np.pi)) * np.sin(n * np.pi * x / L) * np.exp(-alpha * (n * np.pi / L)**2 * t)
    return T_borda + (T_inicial - T_borda) * soma


def conducao_1d_explicita(L, alpha, nx, r, t_final, T_inicial, T_borda):
    """Equação do calor 1D pelo método explícito (FTCS), com as duas pontas a T_borda.

    Devolve um dicionário com:
      x        posições dos nós [m]
      T        temperaturas no instante final [°C]
      t        instante final realmente simulado [s]
      dt       passo de tempo usado [s]
      tempos, T_centro   histórico da temperatura no nó central
    """
    dx = L / (nx - 1)                      # distância entre nós
    dt = r * dx**2 / alpha                 # o passo de tempo sai do número de Fourier r
    n_passos = int(round(t_final / dt))    # quantos passos para chegar a t_final
    x = np.linspace(0, L, nx)

    T = np.full(nx, T_inicial, dtype=float)   # todos os nós começam a T_inicial
    T[0] = T_borda                            # ponta esquerda: temperatura imposta
    T[-1] = T_borda                           # ponta direita (índice -1 = último nó)

    centro = nx // 2                          # // é divisão inteira: índice do nó do meio
    tempos = [0.0]
    T_centro = [T[centro]]

    um_passo = _usar("ftcs")                  # a SUA função da Tarefa B, se passou no verificador; senão, a resposta
    for passo in range(1, n_passos + 1):
        T = um_passo(T, r)                    # um passo do FTCS em TODOS os nós internos (Tarefa B, logo abaixo)
        tempos.append(passo * dt)
        T_centro.append(T[centro])

    return {"x": x, "T": T, "t": n_passos * dt, "dt": dt,
            "tempos": np.array(tempos), "T_centro": np.array(T_centro)}

print("Motor da Parte B carregado.")

### ✍️ Tarefa B — escreva o passo do FTCS

O motor acima chama uma função que avança **um passo de tempo**. Complete-a com a fórmula do FTCS (seção B.1), usando as fatias da Parte 0: `T[1:-1]` (nós internos), `T[2:]` (vizinhos da direita) e `T[:-2]` (vizinhos da esquerda). As bordas não mudam.

O verificador testa o Exemplo B.1 da apostila (`[0; 100; 100; 100; 0]` com $r=0{,}4$) e depois simula a barra inteira contra a série de Fourier. Se travar, abra a resposta logo abaixo.

In [ ]:
def meu_passo_ftcs(T, r):
    T[1:-1] = None   # ← troque None pela fórmula do FTCS
    return T

confere_ftcs(meu_passo_ftcs)

👀 **Resposta da Tarefa B:** a célula abaixo está recolhida. Abra (clique nela ou em *Mostrar código*) só se travar, e rode.

In [ ]:
#@title 👀 Resposta da Tarefa B (abra se travar)
def passo_ftcs_resposta(T, r):
    T[1:-1] = T[1:-1] + r * (T[2:] - 2 * T[1:-1] + T[:-2])
    return T

confere_ftcs(passo_ftcs_resposta, resposta=True)

### Experimento B1 — o passo de tempo e a estabilidade

Barra de alumínio de 10 cm, inicialmente a 100 °C, com as pontas mergulhadas em gelo (0 °C). Simulamos 5 s com uma malha grossa (21 nós) e quatro valores de $r$.

**Antes de rodar:** qual dos casos você espera que dê errado? Como seria esse "errado" no gráfico?

In [ ]:
L, alpha_aluminio = 0.10, 97.1e-6        # comprimento [m] e difusividade do alumínio [m²/s]

# ===== EXPERIMENTO B1: mude os valores de r (número de Fourier da malha) =====
casos = [
    {"nome": "r = 0,25", "r": 0.25},
    {"nome": "r = 0,45", "r": 0.45},
    {"nome": "r = 0,50 (limite)", "r": 0.50},
    {"nome": "r = 0,55 (acima do limite)", "r": 0.55},
]

fig, eixos = plt.subplots(1, len(casos), figsize=(15, 3.6), sharey=True)   # um gráfico por caso
for ax, caso in zip(eixos, casos):
    res = conducao_1d_explicita(L, alpha_aluminio, nx=21, r=caso["r"], t_final=5.0,
                                T_inicial=100.0, T_borda=0.0)
    T_exata = solucao_analitica_barra(res["x"], res["t"], L, alpha_aluminio, 100.0, 0.0)
    ax.plot(res["x"] * 100, T_exata, "k-", lw=1, label="analítica")
    ax.plot(res["x"] * 100, res["T"], "o-", ms=3, label="numérica")
    ax.set_title(f"{caso['nome']}\nΔt = {res['dt']*1000:.0f} ms", fontsize=10)
    ax.set_xlabel("x [cm]"); ax.set_ylim(-20, 120)
eixos[0].set_ylabel("T [°C]"); eixos[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**O que observar.** Até $r=0{,}50$ a solução numérica cola na analítica; em $r=0{,}50$ o método está exatamente no limite, sem nenhuma folga. Acima do limite, um "dente de serra" ponto a ponto cresce a cada passo até dominar tudo: a simulação **diverge**. Não é erro de programação: é o método usado fora do seu limite de estabilidade.

**Sua vez:** experimente $r=0{,}52$ e $t\_final=10$. O que muda?

### Experimento B2 — refinando a malha (convergência)

Agora com $r=0{,}4$ fixo (estável), comparamos malhas de 6 a 41 nós em $t=10$ s.

**Antes de rodar:** ao dobrar o número de intervalos, o erro deve cair quantas vezes? (Dica: lembre da Parte A.)

In [ ]:
# ===== EXPERIMENTO B2: mude as malhas (número de nós) =====
casos = [
    {"nome": "6 nós", "nx": 6},
    {"nome": "11 nós", "nx": 11},
    {"nome": "21 nós", "nx": 21},
    {"nome": "41 nós", "nx": 41},
]

fig, ax = plt.subplots()
x_fino = np.linspace(0, L, 200)
ax.plot(x_fino * 100, solucao_analitica_barra(x_fino, 10.0, L, alpha_aluminio, 100.0, 0.0), "k-", lw=2, label="analítica")
erro_anterior = None
print(" malha   Δx [mm]   erro máximo [°C]   erro anterior / erro atual")
for caso in casos:
    res = conducao_1d_explicita(L, alpha_aluminio, nx=caso["nx"], r=0.4, t_final=10.0,
                                T_inicial=100.0, T_borda=0.0)
    T_exata = solucao_analitica_barra(res["x"], res["t"], L, alpha_aluminio, 100.0, 0.0)
    erro = np.max(np.abs(res["T"] - T_exata))
    razao = f"{erro_anterior / erro:.1f}" if erro_anterior else "—"
    print(f"{caso['nx']:5d}   {res['x'][1]*1000:7.1f}   {erro:14.4f}   {razao:>12}")
    erro_anterior = erro
    ax.plot(res["x"] * 100, res["T"], "o-", ms=4, label=caso["nome"])
ax.set_xlabel("x [cm]"); ax.set_ylabel("T [°C]"); ax.set_title("t = 10 s"); ax.legend()
plt.show()

**O que observar.** A cada refinamento o erro cai cerca de **4 vezes**: o método é de 2ª ordem no espaço (com $r$ fixo, $\Delta t$ cai junto com $\Delta x^2$). É exatamente o "estudo de independência de malha" que qualquer relatório de simulação deve trazer.

### Experimento B3 — o material importa

🎥 **No projetor:** o professor roda este experimento e a turma comenta. Se sobrar tempo, rode você também.

Mesma barra e mesma malha, três materiais. Acompanhamos a temperatura do **centro** ao longo do tempo.

**Antes de rodar:** qual esfria mais rápido? Quantas vezes mais rápido que o aço inox?

In [ ]:
# ===== EXPERIMENTO B3: acrescente materiais (α em m²/s, Incropera, Apêndice A) =====
casos = [
    {"nome": "alumínio", "alpha": 97.1e-6},
    {"nome": "aço carbono", "alpha": 18.8e-6},
    {"nome": "aço inox 304", "alpha": 3.95e-6},
]

fig, ax = plt.subplots()
for caso in casos:
    res = conducao_1d_explicita(L, caso["alpha"], nx=21, r=0.4, t_final=600.0,
                                T_inicial=100.0, T_borda=0.0)
    # tempo para o centro cair abaixo de 50 °C: primeiro instante em que T_centro < 50
    abaixo = np.where(res["T_centro"] < 50.0)[0]      # índices onde a condição é verdadeira
    t50 = res["tempos"][abaixo[0]] if len(abaixo) > 0 else float("nan")
    ax.plot(res["tempos"], res["T_centro"], label=f"{caso['nome']}: 50 °C em {t50:.0f} s")
ax.axhline(50, color="gray", ls=":")
ax.set_xlabel("t [s]"); ax.set_ylabel("T no centro [°C]"); ax.legend()
plt.show()

**O que observar.** O tempo é inversamente proporcional a $\alpha$: o inox ($\alpha$ 25 vezes menor que o do alumínio) demora cerca de 25 vezes mais. A escala de tempo da difusão é $L^2/\alpha$.

### B.3 Condições de contorno: convecção e o nó fantasma

Na prática, a superfície troca calor com um fluido: $-k\,\partial T/\partial x=h\,(T-T_\infty)$. O nó da borda passa a ser incógnita e precisa de um vizinho que não existe, o **nó fantasma** $T_{N+1}$, eliminado pela própria condição de contorno. O resultado (deduzido na apostila) é

$$T_N^{n+1}=T_N^{n}+r\left[2T_{N-1}^{n}-2T_N^{n}-2\,Bi_\Delta\,(T_N^{n}-T_\infty)\right],\qquad Bi_\Delta=\frac{h\,\Delta x}{k}$$

e a estabilidade passa a exigir $r\,(1+Bi_\Delta)\le 1/2$. O motor abaixo já escolhe $r$ respeitando isso.

In [ ]:
def placa_resfriando(L, k, alpha, h, T_inf, T_inicial, nx, t_final):
    """Meia placa de espessura L resfriando por convecção.

    x = 0 é o plano central da placa: por simetria, nenhum calor o atravessa (como se fosse isolado).
    x = L é a superfície, que troca calor com um fluido a T_inf com coeficiente h.
    """
    dx = L / (nx - 1)
    Bi_celula = h * dx / k                   # número de Biot da célula
    r = 0.45 / (1 + Bi_celula)               # passo seguro: r(1 + Bi_celula) = 0,45 < 0,5
    dt = r * dx**2 / alpha
    n_passos = int(round(t_final / dt))
    x = np.linspace(0, L, nx)
    T = np.full(nx, T_inicial, dtype=float)

    for passo in range(n_passos):
        Tn = T.copy()    # cópia do instante atual: as três fórmulas abaixo usam só valores antigos

        # Nós internos: o mesmo FTCS de antes
        T[1:-1] = Tn[1:-1] + r * (Tn[2:] - 2 * Tn[1:-1] + Tn[:-2])

        # Nó 0 (plano central, fluxo nulo): o nó fantasma vale o mesmo que o nó 1,
        # então "vizinho da esquerda + vizinho da direita" vira 2 × (nó 1).
        T[0] = Tn[0] + r * (2 * Tn[1] - 2 * Tn[0])

        # Último nó (superfície com convecção): fórmula do nó fantasma da célula acima.
        T[-1] = Tn[-1] + r * (2 * Tn[-2] - 2 * Tn[-1] - 2 * Bi_celula * (Tn[-1] - T_inf))

    return {"x": x, "T": T, "t": n_passos * dt, "r": r, "dt": dt}

print("Motor da placa com convecção carregado.")

### Experimento B4 — o número de Biot

Placa de aço inox com 10 cm de espessura (simulamos a metade, $L=5$ cm), inicialmente a 200 °C, resfriada por um fluido a 25 °C. Comparamos o perfil de temperatura depois de 10 min para vários $h$. Plotamos a temperatura adimensional $\theta=(T-T_\infty)/(T_i-T_\infty)$.

**Antes de rodar:** com $h$ pequeno o perfil será quase reto (uniforme) ou bem curvo? E com $h$ grande?

In [ ]:
k_inox, alpha_inox = 14.9, 3.95e-6
L_meia = 0.05

# ===== EXPERIMENTO B4: mude os coeficientes de convecção h [W/(m²·K)] =====
casos = [
    {"nome": "ar parado", "h": 10},
    {"nome": "ar soprado", "h": 100},
    {"nome": "água", "h": 1000},
    {"nome": "água em ebulição", "h": 10000},
]

fig, ax = plt.subplots()
for caso in casos:
    res = placa_resfriando(L_meia, k_inox, alpha_inox, caso["h"], T_inf=25.0, T_inicial=200.0,
                           nx=21, t_final=600.0)
    Bi = caso["h"] * L_meia / k_inox
    theta = (res["T"] - 25.0) / (200.0 - 25.0)
    ax.plot(res["x"] * 100, theta, "o-", ms=3, label=f"{caso['nome']} (h = {caso['h']}): Bi = {Bi:.2f}")
ax.set_xlabel("x [cm] (0 = centro, 5 = superfície)"); ax.set_ylabel("θ = (T − T∞)/(Ti − T∞)")
ax.set_ylim(0, 1.05); ax.set_title("Perfis após 10 min"); ax.legend(fontsize=8)
plt.show()

# Modelo de capacitância global (só vale para Bi < 0,1): θ = exp(−h t / (ρ c L))
rho_c = k_inox / alpha_inox
print(f"Capacitância global para h = 10: θ = {np.exp(-10 * 600 / (rho_c * L_meia)):.3f}")

**O que observar.** Com $Bi<0{,}1$ o perfil é praticamente plano: a condução interna é rápida comparada com a convecção, e o modelo de capacitância global funciona (compare o valor impresso com a curva). Com $Bi$ grande, a superfície esfria logo e o centro "fica para trás": aparece um gradiente forte. Repare também que o motor reduziu $r$ automaticamente quando $h$ cresceu, para manter a estabilidade do nó convectivo.

*Para saber mais:* Incropera et al., *Fundamentos de Transferência de Calor e de Massa*, cap. 5 (Biot, capacitância global e diferenças finitas transientes); Özişik, Orlande, Colaço e Cotta, *Finite Difference Methods in Heat Transfer*.

### B.4 Extra de bioengenharia: a equação do biocalor

Um tecido vivo conduz calor como qualquer sólido, mas tem duas fontes a mais: o **sangue**, que chega à temperatura arterial $T_a$ e troca calor com o tecido, e o **metabolismo**. É a equação de Pennes (1948), usada para planejar tratamentos de câncer por hipertermia, criocirurgia e ablação por radiofrequência:

$$\rho c\,\frac{\partial T}{\partial t}=k\,\frac{\partial^{2}T}{\partial x^{2}}+\omega\,\rho_s c_s\,(T_a-T)+\dot q_m$$

Aqui $\omega$ é a **perfusão** (volume de sangue por volume de tecido por segundo). O termo novo não tem derivada: no método explícito, basta somá-lo ao FTCS,

$$T_i^{n+1}=T_i^{n}+r\left(T_{i+1}^{n}-2T_i^{n}+T_{i-1}^{n}\right)+\frac{\Delta t}{\rho c}\left[\omega\,\rho_s c_s\,(T_a-T_i^{n})+\dot q_m\right]$$

Em regime permanente a equação fica $T''-m^{2}(T-T^{*})=0$, com $m^{2}=\omega\rho_s c_s/k$: é a **mesma equação das aletas** (Incropera, cap. 3). O sangue faz o papel da convecção, e $1/m$ é a profundidade até onde uma perturbação na pele consegue penetrar.

In [ ]:
def musculo_com_gelo(omega, t_final, T_pele=5.0, T_fundo=37.0, L=0.03, nx=31,
                     k=0.5, rho_c=3.6e6, rho_c_sangue=3.6e6, T_arterial=37.0, q_met=700.0):
    """Equação do biocalor (Pennes) em 1D pelo método explícito.

    x = 0 é a pele, mantida a T_pele pela bolsa de gelo; x = L é o tecido profundo, a T_fundo.
    omega = perfusão sanguínea [1/s]; q_met = calor do metabolismo [W/m³].
    Propriedades do músculo: Incropera, seção 3.7 (k = 0,5 W/(m·K), omega = 0,0005 1/s, q_met = 700 W/m³).
    """
    dx = L / (nx - 1)
    alpha = k / rho_c
    n_passos = int(np.ceil(t_final / (0.4 * dx**2 / alpha)))   # np.ceil arredonda para cima: garante r <= 0,4
    dt = t_final / n_passos
    r = alpha * dt / dx**2
    x = np.linspace(0, L, nx)
    T = np.full(nx, T_fundo, dtype=float)      # o músculo começa todo a 37 °C
    T[0] = T_pele                              # em t = 0 a bolsa de gelo encosta na pele

    termo_sangue = _usar("sangue")             # a SUA função da Tarefa B5, se passou no verificador; senão, a resposta
    for passo in range(n_passos):
        # termo novo: calor trazido pelo sangue + metabolismo, em K/s (calculado com valores "antigos")
        fonte = termo_sangue(T[1:-1], omega, rho_c, rho_c_sangue, T_arterial, q_met)
        T[1:-1] = T[1:-1] + r * (T[2:] - 2 * T[1:-1] + T[:-2]) + dt * fonte
    return {"x": x, "T": T, "r": r, "dt": dt}


def biocalor_permanente(x, omega, T_pele=5.0, T_fundo=37.0, L=0.03, k=0.5, rho_c_sangue=3.6e6,
                        T_arterial=37.0, q_met=700.0):
    """Solução exata do regime permanente (a mesma das aletas), válida para omega > 0."""
    m = np.sqrt(omega * rho_c_sangue / k)
    T_eq = T_arterial + q_met / (omega * rho_c_sangue)     # temperatura do tecido longe das bordas
    return T_eq + ((T_pele - T_eq) * np.sinh(m * (L - x)) + (T_fundo - T_eq) * np.sinh(m * x)) / np.sinh(m * L)

print("Motor do biocalor carregado.")

### ✍️ Tarefa B5 — o termo do sangue

Complete a função com o termo novo da equação do biocalor, já dividido por $\rho c$ para sair em K/s:

$$\frac{\omega\,\rho_s c_s\,(T_a-T)+\dot q_m}{\rho\,c}$$

`T` são as temperaturas dos nós internos. O verificador confere o termo e depois simula 3 h de gelo contra a solução exata do regime permanente. Se travar, abra a resposta logo abaixo.

In [ ]:
def meu_termo_sangue(T, omega, rho_c, rho_c_sangue, T_arterial, q_met):
    return None   # ← troque None por (omega · rho_c_sangue · (T_arterial − T) + q_met) / rho_c

confere_sangue(meu_termo_sangue)

👀 **Resposta da Tarefa B5:** a célula abaixo está recolhida. Abra (clique nela ou em *Mostrar código*) só se travar, e rode.

In [ ]:
#@title 👀 Resposta da Tarefa B5 (abra se travar)
def termo_sangue_resposta(T, omega, rho_c, rho_c_sangue, T_arterial, q_met):
    return (omega * rho_c_sangue * (T_arterial - T) + q_met) / rho_c

confere_sangue(termo_sangue_resposta, resposta=True)

### Experimento B5 — gelo num músculo lesionado

Uma bolsa de gelo mantém a pele a 5 °C. A 3 cm de profundidade o tecido continua a 37 °C. Comparamos três perfusões depois de 20 min.

**Antes de rodar:** com mais sangue circulando, o frio chega mais fundo ou menos fundo? Por quê?

In [ ]:
# ===== EXPERIMENTO B5: mude a perfusão omega [1/s] ou o tempo de aplicação =====
casos = [
    {"nome": "sem perfusão (circulação interrompida)", "omega": 0.0},
    {"nome": "músculo em repouso", "omega": 0.0005},
    {"nome": "perfusão 10 vezes maior (exercício)", "omega": 0.005},
]
tempo_de_gelo = 20 * 60      # 20 min, em segundos

fig, ax = plt.subplots()
print(f"{'caso':<40}{'T a 1 cm [°C]':>15}{'T a 2 cm [°C]':>15}{'1/m [mm]':>10}{'dif. regime × exata [°C]':>26}")
for caso in casos:
    res = musculo_com_gelo(caso["omega"], tempo_de_gelo)
    linha, = ax.plot(res["x"] * 1000, res["T"], "o-", ms=3, label=caso["nome"])
    if caso["omega"] > 0:
        # verificação: depois de 3 h a simulação deve coincidir com a solução exata do regime permanente
        regime = musculo_com_gelo(caso["omega"], 3 * 3600)
        exata = biocalor_permanente(regime["x"], caso["omega"])
        ax.plot(regime["x"] * 1000, exata, "--", color=linha.get_color(), lw=1)
        penetracao = f"{1000 / np.sqrt(caso['omega'] * 3.6e6 / 0.5):.1f}"
        diferenca = f"{np.max(np.abs(regime['T'] - exata)):.3f}"
    else:
        penetracao, diferenca = "—", "—"
    print(f"{caso['nome']:<40}{res['T'][10]:15.1f}{res['T'][20]:15.1f}{penetracao:>10}{diferenca:>26}")
ax.set_xlabel("profundidade [mm] (0 = pele)"); ax.set_ylabel("T [°C]")
ax.set_title("Depois de 20 min de gelo (tracejado: regime permanente exato)"); ax.legend(fontsize=8)
plt.show()

**O que observar.** O sangue chega a 37 °C e reaquece o tecido: quanto maior a perfusão, **menos fundo** o frio chega. A 1 cm de profundidade, a temperatura depois de 20 min é de cerca de 19 °C sem perfusão, 22 °C em repouso e 32 °C com a perfusão dez vezes maior. A profundidade de penetração $1/m$ cai de 17 mm para 5 mm. As curvas tracejadas são a solução exata do regime permanente: depois de 3 h simuladas, o código coincide com ela (última coluna), a mesma verificação feita com a série de Fourier no Experimento B1.

**Sua vez:** troque `tempo_de_gelo` para 5 e para 40 min. O caso de perfusão alta ainda muda depois de 5 min? O que isso diz sobre o tempo que o tecido leva para "responder"?

*Para saber mais:* Pennes, H. H. *Journal of Applied Physiology*, 1, 1948; Incropera et al., seção 3.7 (equação do biocalor).

## Parte C — Método implícito e algoritmo de Thomas

### C.1 A ideia

O limite $r\le 1/2$ cobra caro: dividir $\Delta x$ por 2 obriga a dividir $\Delta t$ por 4. No **método implícito** (Euler implícito), a difusão é avaliada no instante **novo**:

$$\frac{T_i^{n+1}-T_i^{n}}{\Delta t}=\alpha\,\frac{T_{i+1}^{n+1}-2T_i^{n+1}+T_{i-1}^{n+1}}{\Delta x^{2}}\quad\Longrightarrow\quad -r\,T_{i-1}^{n+1}+(1+2r)\,T_i^{n+1}-r\,T_{i+1}^{n+1}=T_i^{n}$$

Agora cada equação tem **três incógnitas**. Juntas, formam um sistema **tridiagonal**, que tem um método de solução clássico, rápido e simples: o **algoritmo de Thomas** (TDMA). Em troca de resolver esse sistema a cada passo, o método é estável para **qualquer** $\Delta t$.

In [ ]:
def thomas(a, b, c, d):
    """Resolve um sistema tridiagonal pelo algoritmo de Thomas (TDMA).

    Equação de cada linha i:   a[i]*x[i-1] + b[i]*x[i] + c[i]*x[i+1] = d[i]
    (a[0] e c[-1] não são usados: não existe vizinho fora do sistema)
    """
    n = len(d)
    c_mod = np.zeros(n)     # coeficientes modificados durante a eliminação
    d_mod = np.zeros(n)

    # 1) ELIMINAÇÃO PARA FRENTE: linha a linha, eliminamos a incógnita da esquerda.
    #    Depois disso, cada linha fica na forma  x[i] + c_mod[i]*x[i+1] = d_mod[i].
    c_mod[0] = c[0] / b[0]
    d_mod[0] = d[0] / b[0]
    for i in range(1, n):
        denominador = b[i] - a[i] * c_mod[i - 1]
        c_mod[i] = c[i] / denominador if i < n - 1 else 0.0
        d_mod[i] = (d[i] - a[i] * d_mod[i - 1]) / denominador

    # 2) SUBSTITUIÇÃO DE TRÁS PARA FRENTE: a última linha já dá x[n-1] direto;
    #    cada linha anterior usa o valor que acabou de ser encontrado.
    x = np.zeros(n)
    x[-1] = d_mod[-1]
    for i in range(n - 2, -1, -1):          # i = n-2, n-3, ..., 0 (contando para trás)
        x[i] = d_mod[i] - c_mod[i] * x[i + 1]
    return x

# Conferência: comparamos com o solucionador genérico do NumPy num sistema pequeno.
a = np.array([0.0, -1.0, -1.0, -1.0]); b = np.array([4.0, 4.0, 4.0, 4.0])
c = np.array([-1.0, -1.0, -1.0, 0.0]); d = np.array([5.0, 5.0, 10.0, 23.0])
A_cheia = np.diag(b) + np.diag(a[1:], -1) + np.diag(c[:-1], 1)   # monta a matriz completa só para conferir
print("Thomas:      ", thomas(a, b, c, d))
print("NumPy solve: ", np.linalg.solve(A_cheia, d))

In [ ]:
def conducao_1d_implicita(L, alpha, nx, dt, t_final, T_inicial, T_borda):
    """Equação do calor 1D pelo método implícito (Euler implícito) + algoritmo de Thomas."""
    dx = L / (nx - 1)
    r = alpha * dt / dx**2                 # pode ser bem maior que 0,5
    n_passos = int(round(t_final / dt))
    x = np.linspace(0, L, nx)
    T = np.full(nx, T_inicial, dtype=float)
    T[0] = T_borda
    T[-1] = T_borda

    # Coeficientes do sistema tridiagonal (iguais em todos os passos)
    n = nx - 2                             # incógnitas: só os nós internos
    a = np.full(n, -r)                     # vizinho da esquerda
    b = np.full(n, 1 + 2 * r)              # o próprio nó
    c = np.full(n, -r)                     # vizinho da direita
    lado_direito = _usar("lado_direito")   # a SUA função da Tarefa C, se passou no verificador; senão, a resposta

    for passo in range(n_passos):
        d = lado_direito(T, r)             # lado direito: temperaturas atuais + bordas conhecidas (Tarefa C)
        T[1:-1] = thomas(a, b, c, d)       # resolve o sistema e guarda o instante novo

    return {"x": x, "T": T, "t": n_passos * dt, "r": r, "passos": n_passos}

print("Motor implícito carregado.")

### ✍️ Tarefa C — o lado direito do sistema

Na equação do nó $i$, $\;-r\,T_{i-1}^{n+1}+(1+2r)\,T_i^{n+1}-r\,T_{i+1}^{n+1}=T_i^{n}$. No primeiro e no último nó interno, um dos vizinhos é a **borda**, que é conhecida: ela passa para o lado direito somando $r\,T_{\text{borda}}$. Complete as duas linhas. Se travar, abra a resposta logo abaixo.

In [ ]:
def meu_lado_direito(T, r):
    d = T[1:-1].copy()   # temperaturas atuais dos nós internos (uma CÓPIA: não mexe em T)
    d[0] += None         # ← troque None: a borda esquerda, T[0], passa para o lado direito
    d[-1] += None        # ← troque None: a borda direita, T[-1]
    return d

confere_lado_direito(meu_lado_direito)

👀 **Resposta da Tarefa C:** a célula abaixo está recolhida. Abra (clique nela ou em *Mostrar código*) só se travar, e rode.

In [ ]:
#@title 👀 Resposta da Tarefa C (abra se travar)
def lado_direito_resposta(T, r):
    d = T[1:-1].copy()
    d[0] += r * T[0]
    d[-1] += r * T[-1]
    return d

confere_lado_direito(lado_direito_resposta, resposta=True)

### Experimento C1 — passos de tempo grandes

A mesma barra de alumínio (51 nós) até $t=10$ s. O explícito está preso ao limite $\Delta t_{\lim}=\Delta x^2/(2\alpha)$; o implícito pode usar 10 ou 100 vezes esse valor.

**Antes de rodar:** o implícito com $\Delta t$ 100 vezes maior vai explodir? Vai acertar?

In [ ]:
nx = 51
dx = L / (nx - 1)
dt_limite = dx**2 / (2 * alpha_aluminio)       # limite de estabilidade do explícito

# ===== EXPERIMENTO C1: mude os métodos e os passos de tempo =====
casos = [
    {"nome": "explícito, 0,8 × limite", "metodo": "explicito", "dt": 0.8 * dt_limite},
    {"nome": "implícito, 1 × limite", "metodo": "implicito", "dt": 1 * dt_limite},
    {"nome": "implícito, 10 × limite", "metodo": "implicito", "dt": 10 * dt_limite},
    {"nome": "implícito, 100 × limite", "metodo": "implicito", "dt": 100 * dt_limite},
]

fig, ax = plt.subplots()
x_fino = np.linspace(0, L, 200)
ax.plot(x_fino * 100, solucao_analitica_barra(x_fino, 10.0, L, alpha_aluminio, 100.0, 0.0), "k-", lw=2, label="analítica")
print(f"{'caso':<26}{'Δt [s]':>9}{'passos':>8}{'erro máx [°C]':>15}{'tempo [ms]':>12}")
for caso in casos:
    inicio = time.perf_counter()               # relógio antes
    if caso["metodo"] == "explicito":
        r = alpha_aluminio * caso["dt"] / dx**2
        res = conducao_1d_explicita(L, alpha_aluminio, nx, r, 10.0, 100.0, 0.0)
        passos = int(round(10.0 / res["dt"]))
    else:
        res = conducao_1d_implicita(L, alpha_aluminio, nx, caso["dt"], 10.0, 100.0, 0.0)
        passos = res["passos"]
    duracao = (time.perf_counter() - inicio) * 1000   # milissegundos
    erro = np.max(np.abs(res["T"] - solucao_analitica_barra(res["x"], res["t"], L, alpha_aluminio, 100.0, 0.0)))
    print(f"{caso['nome']:<26}{caso['dt']:9.3f}{passos:8d}{erro:15.3f}{duracao:12.1f}")
    ax.plot(res["x"] * 100, res["T"], "o-", ms=3, label=caso["nome"])
ax.set_xlabel("x [cm]"); ax.set_ylabel("T [°C]"); ax.set_title("t = 10 s"); ax.legend(fontsize=8)
plt.show()

**O que observar.** Nenhum caso implícito explode: é a **estabilidade incondicional**. Mas estabilidade não é precisão: com $\Delta t$ muito grande o erro cresce (o Euler implícito é de 1ª ordem no tempo). Na prática, escolhe-se $\Delta t$ pela **precisão** desejada, não mais pela estabilidade. (Com o mesmo $\Delta t$, o implícito gasta mais tempo por passo porque aqui o Thomas está escrito em Python puro, para ficar legível; em código compilado ele é tão rápido quanto o explícito.)

### Experimento C2 — o custo de refinar a malha

🎥 **No projetor:** o professor roda este experimento e a turma comenta. Se sobrar tempo, rode você também.

Para chegar a $t=10$ s, o explícito precisa de cada vez mais passos quando a malha é refinada; o implícito pode manter $\Delta t=0{,}1$ s.

**Antes de rodar:** dobrando o número de nós, o tempo do explícito cresce quantas vezes?

In [ ]:
# ===== EXPERIMENTO C2: mude as malhas =====
malhas = [51, 101, 201]

print(f"{'nós':>5}{'passos explícito':>18}{'tempo explícito [ms]':>22}{'passos implícito':>18}{'tempo implícito [ms]':>22}")
for nx_teste in malhas:
    inicio = time.perf_counter()
    res_exp = conducao_1d_explicita(L, alpha_aluminio, nx_teste, 0.4, 10.0, 100.0, 0.0)
    t_exp = (time.perf_counter() - inicio) * 1000
    inicio = time.perf_counter()
    res_imp = conducao_1d_implicita(L, alpha_aluminio, nx_teste, 0.1, 10.0, 100.0, 0.0)
    t_imp = (time.perf_counter() - inicio) * 1000
    passos_exp = int(round(10.0 / res_exp["dt"]))
    print(f"{nx_teste:5d}{passos_exp:18d}{t_exp:22.1f}{res_imp['passos']:18d}{t_imp:22.1f}")

**O que observar.** O número de passos do explícito cresce 4 vezes a cada vez que a malha dobra (e o trabalho por passo dobra). O implícito mantém o número de passos. Por isso programas de simulação térmica usam, por padrão, métodos implícitos.

*Para saber mais:* Patankar, *Numerical Heat Transfer and Fluid Flow* (1980), seção 4.2 (TDMA); Maliska, *Transferência de Calor e Mecânica dos Fluidos Computacional*.

## Parte D — Condução 2D: transiente e regime permanente

### D.1 A equação em 2D

$$\frac{\partial T}{\partial t}=\alpha\left(\frac{\partial^{2}T}{\partial x^{2}}+\frac{\partial^{2}T}{\partial y^{2}}\right)$$

Cada segunda derivada é discretizada como em 1D (o **estêncil de 5 pontos**). Com $\Delta x=\Delta y$, a estabilidade do explícito passa a exigir $r\le 1/4$: cada nó troca calor com o dobro de vizinhos.

**Indexação:** guardamos $T(x_i,y_j)$ em `T[j, i]` (linha = $y$, coluna = $x$), o formato que os gráficos do Matplotlib esperam.

| Fatia | Nó |
|---|---|
| `T[1:-1, 1:-1]` | $(i,j)$, todos os nós internos |
| `T[1:-1, 2:]` e `T[1:-1, :-2]` | vizinhos em $x$: $(i+1,j)$ e $(i-1,j)$ |
| `T[2:, 1:-1]` e `T[:-2, 1:-1]` | vizinhos em $y$: $(i,j+1)$ e $(i,j-1)$ |

In [ ]:
def placa_com_bordas(n, bordas, T_interior):
    """Cria uma malha n × n com as temperaturas de borda e um valor inicial no interior.
    bordas: dicionário com 'superior', 'inferior', 'esquerda' e 'direita' (°C)."""
    T = np.full((n, n), T_interior, dtype=float)
    T[0, :] = bordas["inferior"]      # linha 0 = y = 0
    T[-1, :] = bordas["superior"]     # última linha = y = L
    T[:, 0] = bordas["esquerda"]      # coluna 0 = x = 0
    T[:, -1] = bordas["direita"]      # última coluna = x = L
    T[-1, 1:-1] = bordas["superior"]  # a borda superior prevalece fora dos cantos
    return T


def conducao_2d_explicita(L, alpha, n, r, t_final, bordas, T_inicial, n_quadros=40):
    """Placa quadrada L × L, método explícito com estêncil de 5 pontos (Δx = Δy).
    Guarda n_quadros instantâneos para fazer gráficos e animações."""
    dx = L / (n - 1)
    dt = r * dx**2 / alpha
    n_passos = int(round(t_final / dt))
    T = placa_com_bordas(n, bordas, T_inicial)
    quadros, tempos = [T.copy()], [0.0]
    guardar_a_cada = max(1, n_passos // n_quadros)
    for passo in range(1, n_passos + 1):
        # FTCS em 2D: soma das segundas diferenças em x (colunas vizinhas) e em y (linhas vizinhas)
        T[1:-1, 1:-1] = (T[1:-1, 1:-1]
                         + r * (T[1:-1, 2:] - 2 * T[1:-1, 1:-1] + T[1:-1, :-2])
                         + r * (T[2:, 1:-1] - 2 * T[1:-1, 1:-1] + T[:-2, 1:-1]))
        if passo % guardar_a_cada == 0:     # % é o resto da divisão: guarda de tantos em tantos passos
            quadros.append(T.copy()); tempos.append(passo * dt)
    return {"T": T, "quadros": quadros, "tempos": tempos, "dt": dt}

print("Motor 2D transiente carregado.")

### Experimento D1 — a placa aquecendo

Placa de alumínio de 10 cm × 10 cm a 20 °C. Em $t=0$ a borda superior passa a 100 °C.

**Antes de rodar:** em quanto tempo você acha que a placa para de mudar?

In [ ]:
# ===== EXPERIMENTO D1: mude as bordas, o material ou o tempo =====
bordas = {"superior": 100.0, "inferior": 20.0, "esquerda": 20.0, "direita": 20.0}
res2d = conducao_2d_explicita(L=0.10, alpha=97.1e-6, n=41, r=0.2, t_final=60.0,
                              bordas=bordas, T_inicial=20.0)

n = 41
X, Y = np.meshgrid(np.linspace(0, 10, n), np.linspace(0, 10, n))   # malha em cm, para os gráficos
mostrar = [1, 4, 12, len(res2d["quadros"]) - 1]                       # quais quadros desenhar
fig, eixos = plt.subplots(1, 4, figsize=(15, 3.5))
for ax, q in zip(eixos, mostrar):
    cf = ax.contourf(X, Y, res2d["quadros"][q], levels=np.linspace(20, 100, 17), cmap="inferno")
    ax.set_title(f"t = {res2d['tempos'][q]:.0f} s"); ax.set_aspect("equal"); ax.set_xlabel("x [cm]")
eixos[0].set_ylabel("y [cm]")
fig.colorbar(cf, ax=eixos, label="T [°C]", shrink=0.85)
plt.show()

In [ ]:
# Animação: FuncAnimation troca a imagem quadro a quadro; to_jshtml cria um player no notebook.
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(5, 4.2))
imagem = ax.imshow(res2d["quadros"][0], origin="lower", extent=[0, 10, 0, 10], cmap="inferno", vmin=20, vmax=100)
fig.colorbar(imagem, label="T [°C]"); ax.set_xlabel("x [cm]"); ax.set_ylabel("y [cm]")

def atualiza(q):
    imagem.set_data(res2d["quadros"][q])
    ax.set_title(f"t = {res2d['tempos'][q]:.0f} s")
    return [imagem]

animacao = FuncAnimation(fig, atualiza, frames=len(res2d["quadros"]), interval=100)
plt.close(fig)               # evita mostrar uma figura estática extra
HTML(animacao.to_jshtml())

**O que observar.** Em poucas dezenas de segundos a placa quase para de mudar: chegou ao **regime permanente**. A escala de tempo do modo mais lento é $L^2/(2\pi^2\alpha)\approx 5$ s. Daqui em diante, interessa só o estado final, $\nabla^2T=0$ (equação de Laplace).

### D.2 Regime permanente: Jacobi, Gauss–Seidel e SOR

Em regime permanente, cada nó é a **média dos quatro vizinhos**: $T_{i,j}=\tfrac{1}{4}\left(T_{i+1,j}+T_{i-1,j}+T_{i,j+1}+T_{i,j-1}\right)$. Isso forma um sistema linear com uma equação por nó. Os métodos iterativos clássicos chutam um valor e corrigem até parar de mudar:

- **Jacobi:** calcula todos os nós novos usando só valores da iteração anterior.
- **Gauss–Seidel:** usa os valores novos assim que são calculados (converge cerca de 2 vezes mais rápido).
- **SOR** (sobrerrelaxação sucessiva): Gauss–Seidel que "exagera" a correção por um fator $\omega$ entre 1 e 2: $T\leftarrow T+\omega\,(\text{média}-T)$.

In [ ]:
def jacobi(T, tol=1e-4, max_iter=50000):
    """Jacobi vetorizado: todos os nós usam os valores da iteração anterior."""
    T = T.copy()
    regra_da_media = _usar("media_jacobi")          # a SUA função da Tarefa D, se passou no verificador; senão, a resposta
    historico = []                                  # maior mudança em cada iteração
    for iteracao in range(max_iter):
        T_antigo = T.copy()
        T[1:-1, 1:-1] = regra_da_media(T_antigo)   # média dos 4 vizinhos (Tarefa D)
        mudanca = np.max(np.abs(T - T_antigo))
        historico.append(mudanca)
        if mudanca < tol:                           # parou de mudar: consideramos convergido
            break
    return T, historico


def gauss_seidel_sor(T, omega=1.0, tol=1e-4, max_iter=50000):
    """Gauss–Seidel (omega = 1) ou SOR (1 < omega < 2), percorrendo nó a nó.
    Cada nó já usa os vizinhos atualizados nesta mesma iteração."""
    T = T.copy()
    ny, nx = T.shape
    corrige = _usar("sor")                         # a SUA correção do SOR (Tarefa D), se passou; senão, a resposta
    historico = []
    for iteracao in range(max_iter):
        maior_mudanca = 0.0
        for j in range(1, ny - 1):                  # percorre as linhas internas
            for i in range(1, nx - 1):              # e, em cada linha, as colunas internas
                media = 0.25 * (T[j, i + 1] + T[j, i - 1] + T[j + 1, i] + T[j - 1, i])
                novo = corrige(T[j, i], media, omega)          # T + ω (média − T); com ω = 1, o novo valor é a própria média
                maior_mudanca = max(maior_mudanca, abs(novo - T[j, i]))
                T[j, i] = novo                      # guardado na hora: o próximo nó já usa este valor
        historico.append(maior_mudanca)
        if maior_mudanca < tol:
            break
    return T, historico

print("Métodos iterativos carregados.")

### ✍️ Tarefa D — Jacobi e SOR

Duas lacunas:

1. a **regra da média** do Jacobi, para todos os nós internos de uma vez, com as fatias 2D da tabela da seção D.1;
2. a **correção do SOR** para um nó: $T\leftarrow T+\omega\,(\text{média}-T)$.

O verificador roda os dois métodos na placa do Experimento D2. Se travar, abra a resposta logo abaixo.

In [ ]:
def minha_media_jacobi(T_antigo):
    # média dos 4 vizinhos em TODOS os nós internos (uma matriz do tamanho de T_antigo[1:-1, 1:-1])
    return None   # ← use T_antigo[1:-1, 2:], T_antigo[1:-1, :-2], T_antigo[2:, 1:-1] e T_antigo[:-2, 1:-1]

def minha_correcao_sor(T_no, media, omega):
    return None   # ← valor novo do nó: T_no + omega × (media − T_no)

confere_jacobi(minha_media_jacobi)
confere_sor(minha_correcao_sor)

👀 **Resposta da Tarefa D:** a célula abaixo está recolhida. Abra (clique nela ou em *Mostrar código*) só se travar, e rode.

In [ ]:
#@title 👀 Resposta da Tarefa D (abra se travar)
def media_jacobi_resposta(T_antigo):
    return 0.25 * (T_antigo[1:-1, 2:] + T_antigo[1:-1, :-2] + T_antigo[2:, 1:-1] + T_antigo[:-2, 1:-1])

def correcao_sor_resposta(T_no, media, omega):
    return T_no + omega * (media - T_no)

confere_jacobi(media_jacobi_resposta, resposta=True)
confere_sor(correcao_sor_resposta, resposta=True)

### Experimento D2 — quem converge mais rápido?

Placa com a borda superior a 100 °C e as demais a 20 °C, malha de 21 × 21 nós. Para cada método, acompanhamos a "maior mudança" a cada iteração.

**Antes de rodar:** qual valor de $\omega$ você acha que é o melhor?

In [ ]:
n = 21
bordas = {"superior": 100.0, "inferior": 20.0, "esquerda": 20.0, "direita": 20.0}
T_chute = placa_com_bordas(n, bordas, T_interior=20.0)       # chute inicial: interior a 20 °C
omega_otimo = 2 / (1 + np.sin(np.pi / (n - 1)))              # fórmula clássica para o quadrado

# ===== EXPERIMENTO D2: mude os métodos e os valores de omega =====
casos = [
    {"nome": "Jacobi", "metodo": "jacobi"},
    {"nome": "Gauss–Seidel (ω = 1)", "metodo": "sor", "omega": 1.0},
    {"nome": "SOR ω = 1,5", "metodo": "sor", "omega": 1.5},
    {"nome": "SOR ω = 1,9", "metodo": "sor", "omega": 1.9},
    {"nome": f"SOR ω ótimo = {omega_otimo:.2f}", "metodo": "sor", "omega": omega_otimo},
]

fig, ax = plt.subplots(figsize=(8, 4))
for caso in casos:
    inicio = time.perf_counter()
    if caso["metodo"] == "jacobi":
        T_final, hist = jacobi(T_chute)
    else:
        T_final, hist = gauss_seidel_sor(T_chute, omega=caso["omega"])
    duracao = time.perf_counter() - inicio
    ax.semilogy(hist, label=f"{caso['nome']}: {len(hist)} iterações")
    print(f"{caso['nome']:<26} {len(hist):6d} iterações   {duracao:6.2f} s")
ax.set_xlabel("iteração"); ax.set_ylabel("maior mudança [°C]"); ax.legend(fontsize=8)
plt.show()

**O que observar.** O Gauss–Seidel precisa de cerca de metade das iterações do Jacobi. O SOR com $\omega$ bem escolhido precisa de cerca de **7 vezes menos** iterações que o Gauss–Seidel (12 vezes menos que o Jacobi) nesta malha, e a vantagem cresce com a malha (Experimento D3). Mas $\omega$ grande demais volta a piorar: existe um ótimo. Repare que o Jacobi vetorizado pode ser mais rápido em segundos mesmo fazendo mais iterações, porque o NumPy faz as contas em código compilado: número de iterações e tempo de computador são coisas diferentes.

### Experimento D3 — e quando a malha cresce?

🎥 **No projetor:** o professor roda este experimento e a turma comenta. Se sobrar tempo, rode você também.

**Antes de rodar:** se a malha dobra, o número de iterações dobra, quadruplica ou fica igual?

In [ ]:
# ===== EXPERIMENTO D3: mude as malhas (malhas grandes deixam o Gauss–Seidel lento) =====
malhas = [11, 21, 31]

print(f"{'malha':>7}{'Jacobi':>10}{'Gauss–Seidel':>15}{'SOR ótimo':>12}")
for n_teste in malhas:
    T0 = placa_com_bordas(n_teste, bordas, 20.0)
    w = 2 / (1 + np.sin(np.pi / (n_teste - 1)))
    _, h_j = jacobi(T0)                          # o "_" descarta o valor que não vamos usar
    _, h_gs = gauss_seidel_sor(T0, omega=1.0)
    _, h_sor = gauss_seidel_sor(T0, omega=w)
    print(f"{n_teste:4d}×{n_teste:<3d}{len(h_j):9d}{len(h_gs):15d}{len(h_sor):12d}")

**O que observar.** Para Jacobi e Gauss–Seidel o número de iterações cresce aproximadamente com o **quadrado** do número de intervalos; para o SOR ótimo, cresce só **proporcionalmente**. Em malhas de engenharia (milhões de nós), isso decide se a simulação leva minutos ou semanas. É por isso que os programas comerciais usam métodos ainda mais rápidos, como o *multigrid*.

### D.3 Solver direto e verificação com a solução analítica

Bibliotecas como o SciPy resolvem o sistema linear diretamente, guardando só os coeficientes não nulos da matriz (matriz **esparsa**). Usamos esse resultado como referência e o comparamos com a solução analítica da placa (Incropera, eq. 4.19):

$$\theta(x,y)=\frac{2}{\pi}\sum_{n=1}^{\infty}\frac{(-1)^{n+1}+1}{n}\,\text{sen}\frac{n\pi x}{L}\,\frac{\text{senh}(n\pi y/L)}{\text{senh}(n\pi W/L)},\qquad\theta=\frac{T-T_1}{T_2-T_1}$$

In [ ]:
import scipy.sparse as sp
from scipy.sparse.linalg import spsolve

def laplace_direto(T):
    """Resolve Laplace (cada nó = média dos vizinhos) montando a matriz esparsa do sistema.
    As bordas de T são as condições de contorno; Δx = Δy."""
    T = T.copy()
    n = T.shape[0]
    m = n - 2                                               # nós internos por direção
    # Matriz 1D da segunda diferença (1, -2, 1) e a matriz identidade
    D = sp.diags([1.0, -2.0, 1.0], [-1, 0, 1], shape=(m, m))
    I = sp.identity(m)
    # Matriz 2D: soma das segundas diferenças em x e em y (produto de Kronecker)
    A = (sp.kron(I, D) + sp.kron(D, I)).tocsc()
    # Lado direito: as bordas conhecidas passam para o outro lado com sinal trocado
    b = np.zeros((m, m))
    b[:, 0] -= T[1:-1, 0]; b[:, -1] -= T[1:-1, -1]           # bordas esquerda e direita
    b[0, :] -= T[0, 1:-1]; b[-1, :] -= T[-1, 1:-1]           # bordas inferior e superior
    T[1:-1, 1:-1] = spsolve(A, b.ravel()).reshape(m, m)      # ravel = vetor; reshape = volta a matriz
    return T


def placa_analitica(X, Y, T1, T2, n_termos=300):
    """Série de Fourier da placa quadrada unitária com a borda superior a T2 e as demais a T1."""
    theta = np.zeros_like(X)
    for n in range(1, n_termos + 1, 2):                     # só os termos ímpares
        a = n * np.pi
        # senh(a·y)/senh(a) escrito de forma que não estoura para n grande
        razao = np.exp(a * (Y - 1)) * (1 - np.exp(-2 * a * Y)) / (1 - np.exp(-2 * a))
        theta += (4 / (n * np.pi)) * np.sin(a * X) * razao
    return T1 + (T2 - T1) * theta

n = 51
T_direto = laplace_direto(placa_com_bordas(n, bordas, 20.0))
Xa, Ya = np.meshgrid(np.linspace(0, 1, n), np.linspace(0, 1, n))   # coordenadas adimensionais
T_exata = placa_analitica(Xa, Ya, 20.0, 100.0)
print(f"Temperatura no centro: {T_direto[n//2, n//2]:.2f} °C (por simetria, a resposta exata é 40 °C)")
print(f"Maior diferença para a solução analítica (fora das bordas): {np.max(np.abs(T_direto - T_exata)[1:-1, 1:-1]):.2f} °C")

# Isotermas e linhas de fluxo de calor q = -k ∇T (sempre perpendiculares às isotermas)
dTdy, dTdx = np.gradient(T_direto)          # derivadas ao longo das linhas (y) e das colunas (x)
fig, ax = plt.subplots(figsize=(6, 5))
cf = ax.contourf(Xa * 10, Ya * 10, T_direto, levels=np.linspace(20, 100, 17), cmap="inferno")
ax.streamplot(Xa * 10, Ya * 10, -dTdx, -dTdy, color="white", density=1.2, linewidth=0.8)
fig.colorbar(cf, label="T [°C]"); ax.set_aspect("equal"); ax.set_xlabel("x [cm]"); ax.set_ylabel("y [cm]")
ax.set_title("Isotermas e linhas de fluxo de calor")
plt.show()

**O que observar.** O centro vale 40 °C: somando quatro placas iguais, cada uma com uma borda quente diferente, obtém-se uma placa toda a 100 °C, então cada uma contribui com 1/4 no centro. A maior diferença para a analítica aparece nos cantos superiores, onde 100 °C encosta em 20 °C (uma singularidade que nenhuma malha resolve totalmente).

**Sua vez:** no Experimento D1 ou aqui, mude o dicionário `bordas` (por exemplo, duas bordas quentes) e rode de novo.

## Fechamento do Dia 1

| Conceito | O que você viu | Onde aparece nos programas comerciais |
|---|---|---|
| Ordem de precisão | o erro cai com $\Delta x$ ou $\Delta x^2$ | estudo de independência de malha |
| Estabilidade | o explícito diverge se $r>1/2$ | limites de passo de tempo, "a solução divergiu" |
| Implícito + sistema linear | $\Delta t$ grande sem explodir | padrão em análise térmica transiente |
| Métodos iterativos | Jacobi < Gauss–Seidel < SOR | solvers lineares, resíduos e critérios de convergência |

### Para saber mais

- Incropera, F. P. et al. *Fundamentos de Transferência de Calor e de Massa*. LTC (caps. 4 e 5: métodos de diferenças finitas).
- Chapra, S. C.; Canale, R. P. *Métodos Numéricos para Engenharia*. AMGH (Gauss–Seidel e relaxação).
- Patankar, S. V. *Numerical Heat Transfer and Fluid Flow*. Hemisphere, 1980.
- Maliska, C. R. *Transferência de Calor e Mecânica dos Fluidos Computacional*. LTC.
- Özişik, M. N.; Orlande, H. R. B.; Colaço, M. J.; Cotta, R. M. *Finite Difference Methods in Heat Transfer*. CRC Press, 2017.
- Gratuito, em português: *Cálculo Numérico — Um Livro Colaborativo* (versão Python), projeto REAMAT/UFRGS: https://www.ufrgs.br/reamat/CalculoNumerico/
- Inspiração da oficina: L. A. Barba e G. F. Forsyth, *CFD Python* (2018).